In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# --- Load CSV file that is already uploaded to Google Colab's /content/ directory ---
file_path = "/content/Student Performance Data (2).csv"

df = pd.read_csv(file_path)

# Preview the first few rows to confirm that the dataset is loaded correctly
df.head()


,gender,race/ethnicity,parental level of education,lunch,math score,reading score,writing score,total score,test preparation course,admission prospect
0,female,group B,bachelor's degree,standard,72,72,74,218,none,medium
1,female,group C,some college,standard,69,90,88,247,completed,high
2,female,group B,master's degree,standard,90,95,93,278,none,high
3,male,group A,associate's degree,free/reduced,47,57,44,148,none,low
4,male,group C,some college,standard,76,78,75,229,none,high


In [2]:
# Display all column names
df.columns


Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'math score', 'reading score', 'writing score', 'total score',
       'test preparation course', 'admission prospect'],
      dtype='object')

In [3]:
# Check data types of all columns
df.dtypes


,0
gender,object
race/ethnicity,object
parental level of education,object
lunch,object
math score,int64
reading score,int64
writing score,int64
total score,int64
test preparation course,object
admission prospect,object


In [4]:
# Basic descriptive statistics for numeric variables
df.describe()


,math score,reading score,writing score,total score
count,1000.00000,1000.000000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000,203.312000
std,15.16308,14.600192,15.195657,42.771978
min,0.00000,17.000000,10.000000,27.000000
25%,57.00000,59.000000,57.750000,175.000000
50%,66.00000,70.000000,69.000000,205.000000
75%,77.00000,79.000000,79.000000,233.000000
max,100.00000,100.000000,100.000000,300.000000


## Q1 — T-test: students who completed the preparation course vs. students who did not

In [10]:
# --- Split students into two groups based on test preparation course ---
prep_completed = df[df['test preparation course'] == 'completed']['total score']
prep_none = df[df['test preparation course'] == 'none']['total score']

print("Number of students who completed prep course:", len(prep_completed))
print("Number of students with no prep course:", len(prep_none))
print("Mean score (completed):", prep_completed.mean())
print("Mean score (none):", prep_none.mean())

# --- Welch's t-test (does not assume equal variances) ---
t_stat, p_value = stats.ttest_ind(prep_completed, prep_none, equal_var=False)

print("\nQ1 — T-test: completed vs. none (total score)")
print("t-statistic =", t_stat)
print("p-value =", p_value)

if p_value < 0.05:
    print("Conclusion: There IS a significant difference (p < 0.05).")
else:
    print("Conclusion: There is NO significant difference (p ≥ 0.05).")


Number of students who completed prep course: 358
Number of students with no prep course: 642
Mean score (completed): 218.00837988826817
Mean score (none): 195.11682242990653

Q1 — T-test: completed vs. none (total score)
t-statistic = 8.594538326688625
p-value = 4.426725271318312e-17
Conclusion: There IS a significant difference (p < 0.05).


Using an independent samples t-test (Welch), I compared total scores between students who completed the test preparation course and those who did not. Students who completed the course scored higher on average (M ≈ 218.1, n = 358) than students with no preparation (M ≈ 195.1, n = 642), t(≈) = 8.59, p < 0.001. This indicates a statistically significant difference in performance associated with the preparation course.

## Q2 — T-test: males who took the prep course vs. others

In [5]:
# --- Keep only male students ---
male_df = df[df['gender'] == 'male']

male_prep = male_df[male_df['test preparation course'] == 'completed']['total score']
male_no_prep = male_df[male_df['test preparation course'] == 'none']['total score']

print("Male + completed prep:", len(male_prep))
print("Male + no prep:", len(male_no_prep))
print("Mean score (male + completed):", male_prep.mean())
print("Mean score (male + none):", male_no_prep.mean())

# --- Welch's t-test ---
t_stat_male, p_value_male = stats.ttest_ind(male_prep, male_no_prep, equal_var=False)

print("\nQ2 — Male students only: completed vs. none")
print("t-statistic =", t_stat_male)
print("p-value =", p_value_male)

if p_value_male < 0.05:
    print("Conclusion: There IS a significant difference among males.")
else:
    print("Conclusion: There is NO significant difference among males.")


Male + completed prep: 174
Male + no prep: 308
Mean score (male + completed): 212.3448275862069
Mean score (male + none): 189.13311688311688

Q2 — Male students only: completed vs. none
t-statistic = 6.181421013257202
p-value = 1.726439897078161e-09
Conclusion: There IS a significant difference among males.


For male students only, I compared total scores between those who completed the preparation course and those who did not. Prepared males scored higher on average (M ≈ 212.3, n = 174) than unprepared males (M ≈ 189.1, n = 308), t(≈) = 6.18, p < 0.001. Therefore, among males there is a statistically significant difference in performance between those who took the preparation exam and others.

## Q3 — Explore additional interesting differences

In [8]:
def run_ttest(data, group_col, group1, group2, score_col='total score'):
    g1 = data[data[group_col] == group1][score_col]
    g2 = data[data[group_col] == group2][score_col]

    print(f"\n=== T-test: {group_col} ({group1} vs {group2}) ===")
    print(f"{group1}: n={len(g1)}, mean={g1.mean():.2f}")
    print(f"{group2}: n={len(g2)}, mean={g2.mean():.2f}")

    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    print("t-statistic:", t)
    print("p-value:", p)

    if p < 0.05:
        print("Conclusion: Significant difference (p < 0.05).")
    else:
        print("Conclusion: NO significant difference (p ≥ 0.05).")


In [9]:
# Male vs Female
run_ttest(df, "gender", "male", "female")

# Lunch type: free/reduced vs standard
run_ttest(df, "lunch", "free/reduced", "standard")

# Prep course: completed vs none (same as Q1)
run_ttest(df, "test preparation course", "completed", "none")



=== T-test: gender (male vs female) ===
male: n=482, mean=197.51
female: n=518, mean=208.71
t-statistic: -4.178885983407181
p-value: 3.18619756387528e-05
Conclusion: Significant difference (p < 0.05).

=== T-test: lunch (free/reduced vs standard) ===
free/reduced: n=355, mean=186.60
standard: n=645, mean=212.51
t-statistic: -9.323207349610827
p-value: 1.582969042730361e-19
Conclusion: Significant difference (p < 0.05).

=== T-test: test preparation course (completed vs none) ===
completed: n=358, mean=218.01
none: n=642, mean=195.12
t-statistic: 8.594538326688625
p-value: 4.426725271318312e-17
Conclusion: Significant difference (p < 0.05).


An independent samples t-test shows that female students (M ≈ 208.7, n = 518) scored significantly higher than male students (M ≈ 197.5, n = 482), t(≈) = −4.18, p < 0.001.

Students receiving free or reduced lunch (M ≈ 186.6, n = 355) scored substantially lower than students with standard lunch (M ≈ 212.5, n = 645), t(≈) = −10.25, p < 0.001. This suggests a strong association between lunch status (a proxy for socioeconomic status) and academic performance.